In [ ]:
import os
from pathlib import Path
rega_root = Path(os.environ["REGA_SEPARATE_ROOT"])

# LDSC enrichment barplots for candidate REGA modules

This notebook creates two publication-ready figures:

1. All candidate modules, grouped by region with subtle region-colored backgrounds.
2. Candidate modules with nominally significant positive GWAS enrichment (`Enrichment > 1` and `Enrichment_p < 0.05`).

Bars are filled by cell type and have no region-colored frames. The significance threshold can be changed in the configuration cell.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string(index=False) if hasattr(obj, "to_string") else obj)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 25,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 15,
    "legend.fontsize": 18,
    "legend.title_fontsize": 18,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
# -------------------------
# Configuration
# -------------------------
tsv_path = rega_root / "04_ldsc/results/h2/AD_Bellenguez_single_module_extend2kb_LDSC_results.tsv"
selected_modules_path = rega_root / "02_run_rega/pathology_association/selected_module.txt"

output_dir = Path("LDSC_candidate_module_barplots")
output_dir.mkdir(parents=True, exist_ok=True)

# 'GWAS enriched' definition for the filtered figure.
significance_column = "Enrichment_p"  # Change to 'FDR_BH_Enrichment_p' for FDR filtering.
alpha = 0.05
require_enrichment_gt_one = True

region_order = ["AG", "EC", "HC", "MTC", "PFC", "TH"]
celltype_order = ["Exc", "Inh", "Ast", "Oli", "OPC", "Mic_Immune"]

region_colors = {
    "AG": "#8DA0CB", "EC": "#FC8D62", "HC": "#66C2A5",
    "MTC": "#E78AC3", "PFC": "#B3B3B3", "TH": "#A6D854",
}

celltype_colors = {
    "Exc": "#0072B2", "Inh": "#E69F00", "Ast": "#009E73",
    "Oli": "#D55E00", "OPC": "#CC79A7",
    "Mic_Immune": "#6B5B53",
}

print(f"Using LDSC table: {tsv_path}")
print(f"Output directory: {output_dir.resolve()}")

In [ ]:
def read_candidate_modules(path):
    if not path.exists():
        raise FileNotFoundError(path)

    pattern = re.compile(r"(?:Mic_Immune|Exc|Inh|Ast|Oli|OPC)_(?:AG|EC|HC|MTC|PFC|TH)_M\d+")
    modules = []
    for line in path.read_text().splitlines():
        match = pattern.search(line)
        if match and match.group(0) not in modules:
            modules.append(match.group(0))
    if not modules:
        raise ValueError(f"No module IDs could be parsed from {path}")
    print(f"Read {len(modules)} candidate modules from {path}")
    return modules

candidate_modules = read_candidate_modules(selected_modules_path)
candidate_rank = {module_id: i for i, module_id in enumerate(candidate_modules)}
candidate_modules

In [ ]:
# Read and validate the LDSC results.
ldsc = pd.read_csv(tsv_path, sep="\t")
if "module_id" not in ldsc.columns:
    if "Category" in ldsc.columns:
        ldsc["module_id"] = ldsc["Category"].astype(str)
    else:
        raise KeyError("The LDSC table needs a 'module_id' or 'Category' column.")

required_columns = {"module_id", "Enrichment", "Enrichment_std_error", significance_column}
missing_columns = required_columns.difference(ldsc.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

numeric_columns = ["Enrichment", "Enrichment_std_error", "Enrichment_p", "FDR_BH_Enrichment_p"]
for column in numeric_columns:
    if column in ldsc.columns:
        ldsc[column] = pd.to_numeric(ldsc[column], errors="coerce")

module_pattern = re.compile(
    r"^(?P<celltype>Mic_Immune|Exc|Inh|Ast|Oli|OPC|Vasc)_"
    r"(?P<region>AG|EC|HC|MTC|PFC|TH)_(?P<module>M\d+)$"
)
parsed = ldsc["module_id"].astype(str).str.extract(module_pattern)
for column in ["celltype", "region", "module"]:
    ldsc[column] = parsed[column]

unparsed = ldsc.loc[ldsc[["celltype", "region", "module"]].isna().any(axis=1), "module_id"].tolist()
if unparsed:
    raise ValueError(f"Could not parse these module IDs: {unparsed}")

duplicates = ldsc.loc[ldsc["module_id"].duplicated(keep=False), "module_id"].unique().tolist()
if duplicates:
    raise ValueError(f"Duplicate module IDs in LDSC table: {duplicates}")

missing_candidates = [module_id for module_id in candidate_modules if module_id not in set(ldsc["module_id"])]
if missing_candidates:
    raise ValueError(f"Candidate modules missing from the LDSC table: {missing_candidates}")

plot_data = ldsc.loc[ldsc["module_id"].isin(candidate_modules)].copy()
plot_data["region_rank"] = plot_data["region"].map({x: i for i, x in enumerate(region_order)})
plot_data["candidate_rank"] = plot_data["module_id"].map(candidate_rank)
plot_data = plot_data.sort_values(["region_rank", "candidate_rank"]).reset_index(drop=True)

print(f"LDSC rows: {len(ldsc)}")
print(f"Candidate modules retained: {len(plot_data)}")
display(plot_data[["module_id", "celltype", "region", "Enrichment", "Enrichment_std_error", "Enrichment_p", "FDR_BH_Enrichment_p"]])

In [ ]:
def p_to_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def plot_ldsc_bars(data, title, basename):
    if data.empty:
        print(f"No rows satisfy the filter for: {title}")
        return None

    data = data.copy().reset_index(drop=True)
    x = np.arange(len(data))
    width = max(10.5, 0.62 * len(data) + 3.8)
    fig, ax = plt.subplots(figsize=(width, 8.2))

    # Region is encoded only by a subtle background band.
    for region in region_order:
        positions = np.flatnonzero(data["region"].eq(region).to_numpy())
        if len(positions) == 0:
            continue
        left, right = positions.min() - 0.5, positions.max() + 0.5
        ax.axvspan(left, right, color=region_colors[region], alpha=0.11, linewidth=0, zorder=0)

    bar_colors = data["celltype"].map(celltype_colors).tolist()
    ax.bar(
        x, data["Enrichment"],
        yerr=data["Enrichment_std_error"],
        color=bar_colors, edgecolor="none", linewidth=0,
        error_kw={"ecolor": "#4D4D4D", "elinewidth": 1.0, "capsize": 3, "capthick": 1.0},
        zorder=3,
    )
    ax.axhline(0, color="#4D4D4D", linewidth=0.8, zorder=2)

    lower = (data["Enrichment"] - data["Enrichment_std_error"]).min()
    upper = (data["Enrichment"] + data["Enrichment_std_error"]).max()
    span = max(upper - min(lower, 0), 1.0)
    y_min = min(0, lower) - 0.08 * span
    y_max = upper + 0.22 * span
    ax.set_ylim(y_min, y_max)

    # Region labels are centered above their corresponding background bands.
    label_y = upper + 0.14 * span
    for region in region_order:
        positions = np.flatnonzero(data["region"].eq(region).to_numpy())
        if len(positions):
            ax.text(positions.mean(), label_y, region, ha="center", va="center",
                    color=region_colors[region], fontweight="bold", fontsize=16)

    # Stars always reflect nominal Enrichment_p, matching the reference figure.
    for i, row in data.iterrows():
        stars = p_to_stars(row.get("Enrichment_p", np.nan))
        if stars:
            star_y = row["Enrichment"] + row["Enrichment_std_error"] + 0.035 * span
            ax.text(i, star_y, stars, ha="center", va="bottom", fontweight="bold", fontsize=16)

    ax.set_xticks(x)
    ax.set_xticklabels(data["module_id"], rotation=90, ha="center", va="top")
    ax.set_xlabel("Cell type–region–module", fontweight="bold")
    ax.set_ylabel("Heritability enrichment", fontweight="bold")
    #ax.set_title(title, fontweight="bold", pad=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.6, alpha=0.7, zorder=1)
    ax.set_axisbelow(True)

    present_celltypes = [ct for ct in celltype_order if ct in set(data["celltype"])]
    handles = [Patch(facecolor=celltype_colors[ct], edgecolor="none", label=ct) for ct in present_celltypes]
    ax.legend(handles=handles, title="Cell type", frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")

    fig.text(0.01, 0.01, "* p < 0.05   ** p < 0.01 (Enrichment_p)", fontsize=15, style="italic")
    fig.tight_layout(rect=[0, 0.045, 0.88, 1])

    png_path = output_dir / f"{basename}.png"
    pdf_path = output_dir / f"{basename}.pdf"
    fig.savefig(png_path, dpi=400, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")
    plt.show()
    return fig

all_candidates_fig = plot_ldsc_bars(
    plot_data,
    "LDSC Heritability Enrichment — AD (Bellenguez) Candidate Modules (extend 2 kb)",
    "AD_Bellenguez_candidate_modules_LDSC_barplot",
)

In [ ]:
# Keep candidate modules with positive and statistically supported GWAS enrichment.
enriched_mask = plot_data[significance_column].lt(alpha)
if require_enrichment_gt_one:
    enriched_mask &= plot_data["Enrichment"].gt(1)

enriched_candidates = plot_data.loc[enriched_mask].copy().reset_index(drop=True)

print(
    f"GWAS-enriched candidate modules: {len(enriched_candidates)} "
    f"({significance_column} < {alpha}"
    + (" and Enrichment > 1)" if require_enrichment_gt_one else ")")
)
display(enriched_candidates[["module_id", "Enrichment", "Enrichment_std_error", "Enrichment_p", "FDR_BH_Enrichment_p"]])

filter_label = "nominal p < 0.05" if significance_column == "Enrichment_p" else "FDR < 0.05"
enriched_fig = plot_ldsc_bars(
    enriched_candidates,
    f"AD GWAS-Enriched Candidate Modules ({filter_label})",
    "AD_Bellenguez_GWAS_enriched_candidate_modules_LDSC_barplot",
)

enriched_candidates.to_csv(
    output_dir / "AD_Bellenguez_GWAS_enriched_candidate_modules.tsv",
    sep="\t", index=False,
)